# Generative AI Foundations

**Objectives**
1. LLM fundamentals: tokens, context windows, Transformers
2. Model types, hallucinations, bias, limitations, responsible AI
3. Core architecture of a GenAI app + dev environment setup
4. Compare models and experiment with prompts

Runs on free Colab (CPU is fine). No API keys needed.

## 0. Environment setup

In [1]:
!pip install -q transformers torch tiktoken pandas sentence-transformers

import time, warnings
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, set_seed

warnings.filterwarnings('ignore')
set_seed(42)
device = 0 if torch.cuda.is_available() else -1
print('GPU available:', torch.cuda.is_available())

d:\miniconda3\envs\genai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: False


## 1. LLM fundamentals

### 1.1 Tokens
LLMs do not read characters or words. They read **tokens** (sub-word pieces mapped to integer IDs).

In [2]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')
texts = [
    'Hello world',
    'Generative AI is transforming software development.',
    'Unbelievably, tokenization splits rare words: antidisestablishmentarianism',
    'नमस्ते दुनिया',
]
for t in texts:
    ids = enc.encode(t)
    print(f'{len(t):>3} chars -> {len(ids):>3} tokens | {[enc.decode([i]) for i in ids]}')

 11 chars ->   2 tokens | ['Hello', ' world']
 51 chars ->   8 tokens | ['Gener', 'ative', ' AI', ' is', ' transforming', ' software', ' development', '.']
 74 chars ->  16 tokens | ['Un', 'belie', 'vably', ',', ' token', 'ization', ' splits', ' rare', ' words', ':', ' ant', 'idis', 'establish', 'ment', 'arian', 'ism']
 13 chars ->  13 tokens | ['न', 'म', 'स', '्�', '�', 'े', ' �', '�', 'ु', 'न', 'ि�', '�', 'ा']


Note how non-English text and rare words cost more tokens. Tokens drive **cost, speed, and context usage**.

### 1.2 Context window
The context window is the maximum number of tokens (prompt + output) the model can attend to at once. Anything beyond it is truncated or rejected.

In [3]:
tok = AutoTokenizer.from_pretrained('gpt2')
print('GPT-2 max context:', tok.model_max_length, 'tokens')

long_text = 'The quick brown fox jumps over the lazy dog. ' * 200
n = len(tok.encode(long_text))
print('Input length:', n, 'tokens ->', 'EXCEEDS window' if n > tok.model_max_length else 'fits')

# Simple strategy: keep the most recent tokens
def fit_to_window(text, max_tokens):
    ids = tok.encode(text)[-max_tokens:]
    return tok.decode(ids)

trimmed = fit_to_window(long_text, 512)
print('After trimming:', len(tok.encode(trimmed)), 'tokens')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2001 > 1024). Running this sequence through the model will result in indexing errors


GPT-2 max context: 1024 tokens
Input length: 2001 tokens -> EXCEEDS window
After trimming: 512 tokens


### 1.3 Transformers
Transformers use **self-attention**: every token computes how much to attend to every other token, in parallel. Stacked layers refine token representations; the final layer predicts the next token.

In [ ]:
model = AutoModelForCausalLM.from_pretrained('gpt2')
cfg = model.config
params = sum(p.numel() for p in model.parameters())
print(f'Layers: {cfg.n_layer} | Heads/layer: {cfg.n_head} | Embedding dim: {cfg.n_embd}')
print(f'Vocab size: {cfg.vocab_size} | Context: {cfg.n_positions} | Params: {params/1e6:.0f}M')

# Next-token prediction: the core of every LLM
prompt = 'The capital of France is'
inputs = tok(prompt, return_tensors='pt')
with torch.no_grad():
    logits = model(**inputs).logits[0, -1]
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f'{tok.decode(i):>12}  {p.item():.3f}')

An LLM is a next-token probability machine. Generation = repeatedly sample from this distribution.

## 2. Types of GenAI models

| Type | Example | Task |
|---|---|---|
| Decoder-only (causal LM) | GPT-2, Llama, Claude | Open-ended text generation, chat |
| Encoder-decoder (seq2seq) | Flan-T5 | Translation, summarization, instruction following |
| Encoder-only | BERT | Classification, embeddings, fill-mask |
| Embedding models | MiniLM | Semantic search, RAG |
| Diffusion models | Stable Diffusion | Image generation |
| Multimodal | GPT-4o, Gemini, Claude | Text + image/audio |

In [ ]:
# Decoder-only: free text continuation
gen = pipeline('text-generation', model='gpt2', device=device)
print(gen('Generative AI can help students by', max_new_tokens=30, do_sample=True)[0]['generated_text'])

# Encoder-decoder: instruction following
t2t = pipeline('text2text-generation', model='google/flan-t5-base', device=device)
print(t2t('Translate English to German: How are you?')[0]['generated_text'])

# Encoder-only: classification
clf = pipeline('sentiment-analysis', device=device)
print(clf('This course on GenAI is excellent!'))

In [ ]:
# Embedding model: semantic similarity
from sentence_transformers import SentenceTransformer, util

emb = SentenceTransformer('all-MiniLM-L6-v2')
sentences = ['How do I reset my password?', 'I forgot my login credentials', 'What is the weather today?']
vecs = emb.encode(sentences)
print(util.cos_sim(vecs, vecs).numpy().round(2))

### 2.1 Hallucinations
Models generate **plausible** text, not **verified** text. Ask about something that does not exist and see whether the model pushes back.

In [ ]:
q = 'Who won the 2031 Cricket World Cup final and what was the score?'
out = gen(q, max_new_tokens=40, do_sample=True, num_return_sequences=3)
for i, o in enumerate(out, 1):
    print(f'--- Sample {i} ---')
    print(o['generated_text'])
# Each sample invents different, confident-sounding 'facts'. That is hallucination.

**Mitigations:** grounding with retrieved documents (RAG), citing sources, lowering temperature, verification steps, human review.

### 2.2 Bias
Models absorb statistical patterns from training data, including stereotypes.

In [ ]:
fill = pipeline('fill-mask', model='bert-base-uncased', device=device)
for subject in ['man', 'woman']:
    res = fill(f'The {subject} worked as a [MASK].', top_k=5)
    print(subject, '->', [r['token_str'] for r in res])

Compare the two lists. Differences in predicted occupations reveal learned bias.

### 2.3 Limitations and responsible AI
- **Knowledge cutoff**: no awareness of events after training
- **Hallucination**: confident fabrication
- **Bias**: reflects training data
- **Limited reasoning and arithmetic reliability**
- **Context limits**: cannot hold unlimited text
- **Privacy/security**: prompt injection, data leakage

**Responsible AI principles:** fairness, transparency, accountability, privacy, safety, human oversight.

## 3. Architecture of a GenAI application

```
User input -> Prompt template -> (Retriever / tools) -> LLM -> Output parser / guardrails -> Response
                                     ^ vector DB, APIs
```

Below is a minimal version of each layer.

In [ ]:
# Layer 1: knowledge base + retriever (mini RAG)
docs = [
    'Python is a high-level programming language created by Guido van Rossum.',
    'A Transformer is a neural network architecture based on self-attention.',
    'Tokens are sub-word units that language models use as input.',
]
doc_vecs = emb.encode(docs)

def retrieve(query, k=1):
    scores = util.cos_sim(emb.encode(query), doc_vecs)[0]
    idx = scores.argsort(descending=True)[:k]
    return [docs[i] for i in idx]

# Layer 2: prompt template
TEMPLATE = 'Answer using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:'

# Layer 3: LLM
llm = pipeline('text2text-generation', model='google/flan-t5-base', device=device)

# Layer 4: output handling / guardrail
def postprocess(text):
    text = text.strip()
    return text if text else 'I could not find an answer.'

def ask(question):
    context = ' '.join(retrieve(question))
    prompt = TEMPLATE.format(context=context, question=question)
    return postprocess(llm(prompt, max_new_tokens=50)[0]['generated_text'])

print(ask('What are tokens?'))
print(ask('Who created Python?'))

**Calling a hosted API (optional).** In production you usually call a hosted model. Store the key in Colab Secrets, never in the notebook:

```python
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
msg = client.messages.create(model='<model-name>', max_tokens=200,
                             messages=[{'role': 'user', 'content': 'Explain tokens in one line.'}])
print(msg.content[0].text)
```

## 4. Compare models and experiment with prompts

### 4.1 Model comparison

In [ ]:
model_specs = [
    ('distilgpt2', 'text-generation'),
    ('gpt2', 'text-generation'),
    ('google/flan-t5-small', 'text2text-generation'),
    ('google/flan-t5-base', 'text2text-generation'),
]
test_prompts = [
    'Summarize in one sentence: Transformers use self-attention to process all tokens in parallel.',
    'What is 17 + 25?',
    'Name three uses of generative AI.',
]

rows = []
for name, task in model_specs:
    pipe = pipeline(task, model=name, device=device)
    n_params = sum(p.numel() for p in pipe.model.parameters()) / 1e6
    for p in test_prompts:
        set_seed(42)
        t0 = time.time()
        kwargs = {'max_new_tokens': 40}
        if task == 'text-generation':
            kwargs['pad_token_id'] = pipe.tokenizer.eos_token_id
        out = pipe(p, **kwargs)[0]['generated_text']
        if task == 'text-generation':
            out = out[len(p):]
        rows.append({'model': name, 'params_M': round(n_params), 'prompt': p[:40],
                     'seconds': round(time.time() - t0, 2), 'output': out.strip().replace('\n', ' ')[:90]})

pd.set_option('display.max_colwidth', 100)
df = pd.DataFrame(rows)
df

**What to observe:** instruction-tuned models (Flan-T5) follow tasks; plain GPT-2 only continues text. Bigger models are slower but generally better. Neither is reliable at arithmetic.

### 4.2 Prompt experiments
Same model, different prompts, very different outputs.

In [ ]:
review = 'The battery dies in two hours but the screen is gorgeous.'

prompts = {
    'vague': review,
    'zero-shot': f'Classify the sentiment as positive, negative or mixed: {review}',
    'few-shot': (
        'Review: I love it! -> positive\n'
        'Review: Totally broken. -> negative\n'
        'Review: Great camera, awful price. -> mixed\n'
        f'Review: {review} ->'
    ),
    'role + format': f'You are a product analyst. Give the sentiment (one word) of this review: {review}',
}
for name, p in prompts.items():
    print(f'{name:>14}: {t2t(p, max_new_tokens=20)[0]["generated_text"]}')

### 4.3 Sampling parameters
- **temperature**: low = focused and repeatable, high = diverse and risky
- **top_p / top_k**: restrict sampling to the most likely tokens
- **max_new_tokens**: cap on output length

In [ ]:
prompt = 'The future of artificial intelligence is'
for temp in [0.2, 0.8, 1.5]:
    set_seed(7)
    out = gen(prompt, max_new_tokens=30, do_sample=True, temperature=temp, top_p=0.95,
              pad_token_id=50256)[0]['generated_text']
    print(f'temperature={temp}: {out}\n')

set_seed(7)
greedy = gen(prompt, max_new_tokens=30, do_sample=False, pad_token_id=50256)[0]['generated_text']
print('greedy (deterministic):', greedy)

### 4.4 Your turn
1. Add another model to `model_specs` (e.g. `google/flan-t5-large`) and compare speed vs quality.
2. Write a 5th prompt style (chain-of-thought: add "Let's think step by step") and test it on a word problem.
3. Re-run the bias probe with other words (`nurse`, `engineer`, `doctor`) and record what you find.

## Summary
- LLMs predict the next token; tokens and context windows set cost and limits.
- Different architectures suit different tasks; none are free of hallucination or bias.
- A GenAI app = prompt template + retrieval + model + output handling + guardrails.
- Prompt wording and sampling settings change results as much as model choice.